### Ingest Fact Data into Bronze Layer

### Import Spark types and helpers used during raw fact ingestion.

In [0]:
# Import the dependency used by the lines below.
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F

### Load project config and define a small validation helper for raw order files.

In [0]:
# Load the catalog name from Spark config, or use the default project catalog.
catalog_name = spark.conf.get("training_0002_ecommerce.catalog_name", "training_0002_ecommerce")
source_base_path = spark.conf.get("training_0002_ecommerce.source_base_path", f"/Volumes/{catalog_name}/source_data/raw_data")

# Fail fast if the configured order-item source path produces no records.
# Define a helper that stops the notebook when a required dataset is empty.
def validate_non_empty(df, dataset_name):
    # Count the rows so the dataset can be validated before writing it out.
    row_count = df.count()
    # Check whether the validation condition is met before continuing.
    if row_count == 0:
        # Stop execution with a clear error message when the validation fails.
        raise ValueError(f"{dataset_name} is empty. Check the configured source path before writing Bronze tables.")
    # Print a small status message so the notebook run is easier to follow.
    print(f"Validated {dataset_name}: {row_count} rows")

In [0]:
order_items_schema = StructType([
    StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",        StringType(), True),
])

### Read raw order-item CSV files, attach metadata, and prepare Bronze output.

In [0]:
# Load data using the schema defined
# Point Spark to the raw input file location for this dataset.
raw_data_path = f"{source_base_path}/order_items/*.csv"

# Read the raw file into a DataFrame using the schema defined above.
df = spark.read.option("header", "true").option("delimiter", ",").schema(order_items_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [0]:
validate_non_empty(df, "order_items")
# Show a sample of the result so it can be visually checked.
display(df.limit(5))

### Persist raw order-item data into the Bronze layer.

In [0]:
# Write the current DataFrame to a Delta table in the target layer.
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_order_items")